# Results visualization

Four market-design cases × five risk settings (`RN` + `RA beta 0.2/0.4/0.6/0.8`).

Reads CSVs from the project **`Results/`** folder. Cases or risk settings that have not been generated yet (for example the green social planners) are **skipped**, not treated as errors.

**Run order:** imports → paths → loaders → derived tables → figure cells.

Each figure is saved as PNG **and** a matching CSV table under `visualization_figures/`.

In [ ]:
# Cell 1 — imports
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import yaml
from matplotlib.ticker import PercentFormatter

print("Imports OK")

In [ ]:
# Cell 2 — paths, labels, and plot defaults
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "Data" / "data.yaml").is_file():
            return p
    return here

ROOT = find_project_root()
RESULTS_ROOT = ROOT / "Results"
if not RESULTS_ROOT.is_dir():
    alt = sorted(
        (p for p in ROOT.glob("Results*") if p.is_dir()),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    RESULTS_ROOT = alt[0] if alt else RESULTS_ROOT

FIG_DIR = ROOT / "visualization_figures"
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR = FIG_DIR / "tables"
TABLE_DIR.mkdir(exist_ok=True)

# Canonical risk keys, display labels, and folder-name aliases under Results/
RISK_SPECS = [
    ("RN", "RN", ("RN", "Risk Neutral", "risk_neutral")),
    ("RA beta 0.2", "β=0.2", ("RA beta 0.2", "Beta 0.2", "β=0.2", "beta_0.2")),
    ("RA beta 0.4", "β=0.4", ("RA beta 0.4", "Beta 0.4", "β=0.4", "beta_0.4")),
    ("RA beta 0.6", "β=0.6", ("RA beta 0.6", "Beta 0.6", "β=0.6", "beta_0.6")),
    ("RA beta 0.8", "β=0.8", ("RA beta 0.8", "Beta 0.8", "β=0.8", "beta_0.8")),
]
RISK_ORDER = [key for key, _, _ in RISK_SPECS]
RISK_FOLDERS = {key: label for key, label, _ in RISK_SPECS}
RISK_LABELS = [RISK_FOLDERS[k] for k in RISK_ORDER]

RISK_DIRS: dict[str, Path] = {}
for key, _label, aliases in RISK_SPECS:
    for alias in aliases:
        p = RESULTS_ROOT / alias
        if p.is_dir():
            RISK_DIRS[key] = p
            break
    else:
        RISK_DIRS[key] = RESULTS_ROOT / key


CASE_FOLDERS = {
    "sp": "social_planner_results",
    "me": "market_exposure_results",
    "gsp": "green_social_planner_results",
    "gh2": "green_h2_social_planner_results",
}
CASE_TITLES = {
    "sp": "Social planner (SP)",
    "me": "Market exposure (ME)",
    "gsp": "Green social planner (G-SP)",
    "gh2": "Green H₂ social planner (GH2-SP)",
}

PRICE_COLS = {
    "elec": "Elec_Price",
    "H2": "H2_Price",
    "elec_GC": "Elec_GC_Price",
    "H2_GC": "H2_GC_Price",
    "EP": "EP_Price",
}
MARKET_LABELS = {
    "elec": "Electricity",
    "H2": "Hydrogen",
    "elec_GC": "Electricity GC",
    "H2_GC": "H₂ GC",
    "EP": "Ammonia (EP)",
}

INVESTABLE = [
    "Gen_VRES_Solar",
    "Gen_VRES_Wind",
    "Prod_H2_Green",
    "Offtaker_Green",
]
INVESTABLE_LABELS = {
    "Gen_VRES_Solar": "Solar",
    "Gen_VRES_Wind": "Wind",
    "Prod_H2_Green": "Electrolyzer",
    "Offtaker_Green": "Green EP plant",
}
INVESTABLE_UNITS = {
    "Gen_VRES_Solar": "GW",
    "Gen_VRES_Wind": "GW",
    "Prod_H2_Green": "MW",
    "Offtaker_Green": "MW",
}
INVESTABLE_SCALE = {
    "Gen_VRES_Solar": 1e-3,
    "Gen_VRES_Wind": 1e-3,
    "Prod_H2_Green": 1.0,
    "Offtaker_Green": 1.0,
}

N_TS, N_RD, N_YR = 24, 8, 15

# Palette — muted, print-friendly, distinct without purple glow
C_GREY = "#6B7280"
C_INK = "#1F2937"
C_GRID = "#E5E7EB"
C_FACE = "#FAFAF8"
RISK_COLORS = {
    "RN": "#0F766E",
    "RA beta 0.2": "#1D4ED8",
    "RA beta 0.4": "#B45309",
    "RA beta 0.6": "#C2410C",
    "RA beta 0.8": "#9F1239",
}
MARKET_COLORS = {
    "elec": "#1D4ED8",
    "H2": "#0F766E",
    "elec_GC": "#64748B",
    "H2_GC": "#B45309",
    "EP": "#9F1239",
}
INV_COLORS = {
    "Gen_VRES_Solar": "#CA8A04",
    "Gen_VRES_Wind": "#1D4ED8",
    "Prod_H2_Green": "#0F766E",
    "Offtaker_Green": "#9F1239",
}
CASE_COLORS = {
    "sp": "#0F766E",
    "me": "#1D4ED8",
    "gsp": "#B45309",
    "gh2": "#9F1239",
}

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "figure.facecolor": "white",
    "axes.facecolor": C_FACE,
    "axes.edgecolor": "#D1D5DB",
    "axes.labelcolor": C_INK,
    "axes.titlecolor": C_INK,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.titleweight": "semibold",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": C_GRID,
    "grid.linewidth": 0.8,
    "font.size": 10.5,
    "font.family": "DejaVu Sans",
    "legend.fontsize": 9,
    "legend.frameon": False,
    "xtick.color": C_INK,
    "ytick.color": C_INK,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"ROOT         = {ROOT}")
print(f"RESULTS_ROOT = {RESULTS_ROOT}")
print(f"exists: {RESULTS_ROOT.is_dir()}")
print(f"FIG_DIR      = {FIG_DIR}")
print(f"TABLE_DIR    = {TABLE_DIR}")
print("Risk folders:")
for key in RISK_ORDER:
    p = RISK_DIRS[key]
    print(f"  {key:14s} -> {p.name if p.exists() else '(missing)':20s}  {'OK' if p.is_dir() else 'MISSING'}")
print("Case subfolders:")
for case_key, folder in CASE_FOLDERS.items():
    found = []
    missing = []
    for key in RISK_ORDER:
        d = RISK_DIRS[key] / folder
        (found if d.is_dir() else missing).append(key)
    status = ", ".join(found) if found else "none"
    extra = f"  (missing: {', '.join(missing)})" if missing else ""
    print(f"  {case_key:4s} {folder:36s} {status}{extra}")

In [ ]:
# Cell 3 — YAML helpers (grey ammonia MC + capacity seeds)
with open(ROOT / "Data" / "data.yaml", encoding="utf-8") as f:
    DATA_YAML = yaml.safe_load(f)

fuel = DATA_YAML["Fuel"]
grey = DATA_YAML["Hydrogen_Offtaker"]["Offtaker_Grey"]
gas_mults = [float(x) for x in DATA_YAML["Scenarios"]["gas_price_multipliers"]]
weather_years = [int(x) for x in DATA_YAML["Scenarios"]["weather_years"]]
EP_TOTAL_DEMAND = float(DATA_YAML["EP_market"]["Total_Demand"])
EP_DEMAND_COL = str(DATA_YAML["EP_market"].get("Demand_Column", "LOAD_EP"))

CAPACITY_SEED = {
    "Gen_VRES_Solar": float(DATA_YAML["Power"]["Gen_VRES_Solar"]["Capacity"]),
    "Gen_VRES_Wind": float(DATA_YAML["Power"]["Gen_VRES_Wind"]["Capacity"]),
    "Prod_H2_Green": float(DATA_YAML["Hydrogen"]["Prod_H2_Green"]["Capacity_H2_Output"]),
    "Offtaker_Green": float(DATA_YAML["Hydrogen_Offtaker"]["Offtaker_Green"]["Capacity_EP_Out"]),
}

def grey_ammonia_mc(gas_multiplier: float) -> float:
    return (
        float(grey["GasIntensity"]) * float(fuel["GasPrice"]) * float(gas_multiplier)
        + float(grey["CO2Intensity"]) * float(fuel["CO2Price"])
        + float(grey["VariableOM"])
    )

GREY_MC_BY_GAS = {g: grey_ammonia_mc(g) for g in gas_mults}
GREY_MC_EXPECTED = float(np.mean(list(GREY_MC_BY_GAS.values())))

N_TS = int(DATA_YAML["General"].get("nTimesteps", N_TS))
N_RD = int(DATA_YAML["General"].get("nReprDays", N_RD))
N_YR = len(weather_years) * len(gas_mults)

print("Gas multipliers:", gas_mults)
print(f"Expected grey MC: {GREY_MC_EXPECTED:.2f} EUR/MWh_EP")
print("Capacity seeds (MW):", {k: round(v, 1) for k, v in CAPACITY_SEED.items()})
print(f"Time grid: {N_TS} x {N_RD} x {N_YR} = {N_TS * N_RD * N_YR} slots")

In [ ]:
# Cell 4 — CSV loaders
REQUIRED_CSVS = (
    "Market_Prices.csv",
    "Agent_Objectives_Per_Timestep.csv",
    "Agent_Summary.csv",
)

def run_dir(risk_key: str, case_key: str) -> Path:
    return RISK_DIRS[risk_key] / CASE_FOLDERS[case_key]

def _finite(val) -> bool:
    try:
        return bool(np.isfinite(float(val)))
    except (TypeError, ValueError):
        return False

def load_metrics_csv(path: Path) -> dict:
    df = pd.read_csv(path)
    out = {}
    if "Metric" not in df.columns or "Value" not in df.columns:
        return out
    for _, row in df.iterrows():
        key = str(row["Metric"])
        val = row["Value"]
        try:
            out[key] = float(val)
        except (TypeError, ValueError):
            out[key] = val
    return out

def load_system_metrics(path: Path) -> dict:
    return load_metrics_csv(path)

def _system_cost_from_welfare(case_dir: Path) -> float:
    wpath = case_dir / "Welfare_By_Agent_Per_Year.csv"
    if not wpath.is_file():
        return np.nan
    wdf = pd.read_csv(wpath)
    if "welfare" not in wdf.columns or "probability" not in wdf.columns:
        return np.nan
    if "is_demand" in wdf.columns:
        demand = wdf["is_demand"].astype(str).str.lower().isin(("true", "1", "yes"))
        rest = wdf.loc[~demand]
    else:
        rest = wdf
    return float((-rest["welfare"].astype(float) * rest["probability"].astype(float)).sum())

def load_run_metrics(case_dir: Path) -> dict:
    """Combine Risk_Metrics.csv / System_Metrics.csv and derive system cost."""
    out: dict = {}
    for name in ("System_Metrics.csv", "Risk_Metrics.csv"):
        p = case_dir / name
        if p.is_file():
            out.update(load_metrics_csv(p))
    if not _finite(out.get("expected_total_system_cost")):
        if _finite(out.get("system_cost_expected")):
            out["expected_total_system_cost"] = float(out["system_cost_expected"])
        elif _finite(out.get("expected_welfare_ex_demand")):
            out["expected_total_system_cost"] = -float(out["expected_welfare_ex_demand"])
        else:
            out["expected_total_system_cost"] = _system_cost_from_welfare(case_dir)
    return out

def load_prices_with_weights(case_dir: Path) -> pd.DataFrame:
    prices = pd.read_csv(case_dir / "Market_Prices.csv")
    ao = pd.read_csv(
        case_dir / "Agent_Objectives_Per_Timestep.csv",
        usecols=["Time", "jh", "jd", "jy", "W"],
    )
    df = prices.merge(ao, on="Time", how="left", validate="one_to_one")
    if df["W"].isna().any():
        raise ValueError(f"Missing W after merge in {case_dir}")
    return df

def slot_weights(df: pd.DataFrame) -> np.ndarray:
    w = df["W"].to_numpy(dtype=float)
    n_scen = int(df["jy"].nunique()) if "jy" in df.columns else N_YR
    n_scen = n_scen if n_scen > 0 else N_YR
    return w * (1.0 / n_scen)

def w_mean_price(df: pd.DataFrame, col: str) -> float:
    if col not in df.columns:
        return np.nan
    y = df[col].to_numpy(dtype=float)
    w = df["W"].to_numpy(dtype=float)
    if not np.any(np.isfinite(w)) or float(np.nansum(w)) == 0.0:
        return float(np.nanmean(y))
    mask = np.isfinite(y) & np.isfinite(w)
    if not np.any(mask):
        return np.nan
    return float(np.average(y[mask], weights=w[mask]))

def cwap_from_qty_price(qty: np.ndarray, price: np.ndarray, wt: np.ndarray) -> float:
    q = np.maximum(np.asarray(qty, dtype=float), 0.0)
    p = np.asarray(price, dtype=float)
    w = np.asarray(wt, dtype=float)
    mask = np.isfinite(q) & np.isfinite(p) & np.isfinite(w)
    den = float(np.sum(w[mask] * q[mask]))
    if den <= 1e-12:
        return np.nan
    return float(np.sum(w[mask] * p[mask] * q[mask]) / den)

def ep_qty_vector(ao: pd.DataFrame) -> np.ndarray:
    cols = [c for c in ("Offtaker_Green_ep", "Offtaker_Grey_ep", "Offtaker_Import_ep") if c in ao.columns]
    if not cols:
        return np.full(len(ao), np.nan)
    return ao[cols].to_numpy(dtype=float).sum(axis=1)

def load_investments(case_dir: Path) -> pd.Series:
    inv = pd.Series(np.nan, index=INVESTABLE, dtype=float)
    inv_path = case_dir / "Investable_Capacities.csv"
    summary_path = case_dir / "Agent_Summary.csv"
    summary = (
        pd.read_csv(summary_path).set_index("AgentID")
        if summary_path.is_file()
        else None
    )

    if inv_path.exists():
        df = pd.read_csv(inv_path).set_index("AgentID")
        for aid in INVESTABLE:
            if aid in df.index:
                inv[aid] = float(df.loc[aid, "Investment_Total_MW"])

    if summary is not None:
        for aid in INVESTABLE:
            if pd.notna(inv[aid]):
                continue
            if aid not in summary.index:
                continue
            v = float(summary.loc[aid, "Investment_Total_MW"])
            cap = float(summary.loc[aid, "Capacity_Final_MW"])
            inv[aid] = v if v > 0 else max(0.0, cap - CAPACITY_SEED[aid])

    sp_cap_path = case_dir / "SP_Capacities.csv"
    if sp_cap_path.is_file():
        spc = pd.read_csv(sp_cap_path)
        for aid in INVESTABLE:
            if pd.notna(inv[aid]):
                continue
            hit = spc.loc[spc["AgentID"] == aid, "cap"] if "AgentID" in spc.columns else pd.Series(dtype=float)
            if hit.empty:
                continue
            cap = float(hit.mean())
            inv[aid] = max(0.0, cap - CAPACITY_SEED[aid])

    return inv.fillna(0.0)

def metrics_row(sm: dict, prices_df: pd.DataFrame, ao: pd.DataFrame) -> dict:
    wt = slot_weights(prices_df)
    q_elec = ao["Cons_Elec_01_d"].to_numpy(dtype=float) if "Cons_Elec_01_d" in ao.columns else None
    q_gc = (
        ao["Demand_GC_Elec_01_d_gc"].to_numpy(dtype=float)
        if "Demand_GC_Elec_01_d_gc" in ao.columns
        else None
    )
    if "D_EP_FLAT" in globals() and len(D_EP_FLAT) == len(prices_df):
        q_ep = D_EP_FLAT
    else:
        q_ep = ep_qty_vector(ao)

    cwap_elec = sm.get("consumer_Cons_Elec_01_elec_weighted_avg_price", np.nan)
    if not _finite(cwap_elec) and q_elec is not None:
        cwap_elec = cwap_from_qty_price(q_elec, prices_df["Elec_Price"], wt)

    cwap_gc = sm.get("consumer_Demand_GC_Elec_01_elec_GC_weighted_avg_price", np.nan)
    if not _finite(cwap_gc) and q_gc is not None:
        cwap_gc = cwap_from_qty_price(q_gc, prices_df["Elec_GC_Price"], wt)

    cwap_ep = sm.get("consumer_Ammonia_Demand_EP_weighted_avg_price", sm.get("ep_cwap", np.nan))
    if not _finite(cwap_ep):
        cwap_ep = cwap_from_qty_price(q_ep, prices_df["EP_Price"], wt)

    cost = sm.get("expected_total_system_cost", np.nan)
    return {
        "gamma": sm.get("gamma", np.nan),
        "beta": sm.get("beta", np.nan),
        "system_cost_bn": (float(cost) / 1e9) if _finite(cost) else np.nan,
        "cwap_elec": cwap_elec,
        "cwap_elec_GC": cwap_gc,
        "cwap_EP": cwap_ep,
        **{f"wmean_{k}": w_mean_price(prices_df, c) for k, c in PRICE_COLS.items()},
    }

print("Loader helpers defined")

In [ ]:
# Cell 5 — build inelastic EP demand on the ADMM/SP time grid
def build_ep_demand_mwh() -> np.ndarray:
    n_gas = len(gas_mults)
    out = np.zeros(N_TS * N_RD * N_YR, dtype=float)
    idx = 0
    for weather in weather_years:
        ts = pd.read_csv(ROOT / "Input" / f"timeseries_{weather}.csv")
        profile = ts[EP_DEMAND_COL].to_numpy(dtype=float)
        for _g in range(n_gas):
            for jd in range(N_RD):
                for jh in range(N_TS):
                    row = jd * N_TS + jh
                    out[idx] = EP_TOTAL_DEMAND * float(profile[row])
                    idx += 1
    return out

D_EP_FLAT = build_ep_demand_mwh()
print(f"D_EP flat length={len(D_EP_FLAT)}, mean={D_EP_FLAT.mean():.1f} MW_EP")

In [ ]:
# Cell 6 — import all run CSVs (skip cases/risks that are not generated yet)
system_metrics: dict[str, dict[str, dict]] = {ck: {} for ck in CASE_FOLDERS}
prices: dict[str, dict[str, pd.DataFrame]] = {ck: {} for ck in CASE_FOLDERS}
ao_full: dict[str, dict[str, pd.DataFrame]] = {ck: {} for ck in CASE_FOLDERS}
investments: dict[str, dict[str, pd.Series]] = {ck: {} for ck in CASE_FOLDERS}
summary: dict[str, pd.DataFrame] = {}
inv_tables: dict[str, pd.DataFrame] = {}
skipped_runs: list[str] = []

for case_key in CASE_FOLDERS:
    rows, inv_rows = [], []
    for risk in RISK_ORDER:
        d = run_dir(risk, case_key)
        if not d.is_dir():
            skipped_runs.append(f"{case_key}/{risk}: folder missing ({d.name})")
            continue
        missing = [name for name in REQUIRED_CSVS if not (d / name).is_file()]
        if missing:
            skipped_runs.append(f"{case_key}/{risk}: missing {', '.join(missing)}")
            continue
        try:
            sm = load_run_metrics(d)
            pr = load_prices_with_weights(d)
            ao = pd.read_csv(d / "Agent_Objectives_Per_Timestep.csv")
            inv = load_investments(d)
        except Exception as exc:
            skipped_runs.append(f"{case_key}/{risk}: {type(exc).__name__}: {exc}")
            continue

        system_metrics[case_key][risk] = sm
        prices[case_key][risk] = pr
        ao_full[case_key][risk] = ao
        investments[case_key][risk] = inv

        row = metrics_row(sm, pr, ao)
        row.update(risk_folder=risk, risk_label=RISK_FOLDERS[risk])
        rows.append(row)

        inv_row = {"risk_folder": risk, "risk_label": RISK_FOLDERS[risk]}
        for aid in INVESTABLE:
            inv_row[aid] = float(inv[aid])
        inv_rows.append(inv_row)

        cost_s = f"{row['system_cost_bn']:.3f} bn" if _finite(row["system_cost_bn"]) else "n/a"
        ep_s = f"{row['cwap_EP']:.1f}" if _finite(row["cwap_EP"]) else "n/a"
        print(f"loaded {case_key:4s} | {risk:14s} | cost={cost_s} | EP CWAP={ep_s}")

    if rows:
        summary[case_key] = pd.DataFrame(rows).set_index("risk_folder")
        inv_tables[case_key] = pd.DataFrame(inv_rows).set_index("risk_folder")
    else:
        print(f"no runs loaded for {case_key} ({CASE_TITLES[case_key]})")

AVAILABLE_CASES = [ck for ck in CASE_FOLDERS if ck in summary]
print(f"\nLoaded cases: {', '.join(AVAILABLE_CASES) if AVAILABLE_CASES else 'none'}")
if skipped_runs:
    print(f"Skipped {len(skipped_runs)} run(s):")
    for line in skipped_runs:
        print(f"  - {line}")
print("Done loading.")

In [ ]:
# Cell 7 — total risk-adjusted consumer cost (TRACC)
def scenario_probability(jy: np.ndarray) -> np.ndarray:
    return np.full(jy.shape, 1.0 / N_YR, dtype=float)

def compute_tracc(case_key: str, risk: str) -> dict:
    ao = ao_full[case_key][risk]
    pr = prices[case_key][risk]
    df = ao[["Time", "jh", "jd", "jy", "W", "Cons_Elec_01_d"]].merge(
        pr[["Time", "Elec_Price", "EP_Price"]], on="Time", validate="one_to_one"
    )
    if len(df) != len(D_EP_FLAT):
        raise ValueError(f"Time-grid mismatch for {case_key}/{risk}")

    wt = df["W"].to_numpy(float) * scenario_probability(df["jy"].to_numpy(int))
    q_elec = np.maximum(df["Cons_Elec_01_d"].to_numpy(float), 0.0)
    q_ep = np.maximum(D_EP_FLAT, 0.0)
    λ_elec = df["Elec_Price"].to_numpy(float)
    λ_ep = df["EP_Price"].to_numpy(float)

    exp_elec = float(np.sum(wt * λ_elec * q_elec))
    exp_ep = float(np.sum(wt * λ_ep * q_ep))
    qty_elec = float(np.sum(wt * q_elec))
    qty_ep = float(np.sum(wt * q_ep))
    exp_tot = exp_elec + exp_ep
    qty_tot = qty_elec + qty_ep
    return {
        "exp_elec": exp_elec,
        "exp_ep": exp_ep,
        "exp_total": exp_tot,
        "qty_elec": qty_elec,
        "qty_ep": qty_ep,
        "qty_total": qty_tot,
        "cwap_elec": exp_elec / qty_elec if qty_elec else np.nan,
        "cwap_ep": exp_ep / qty_ep if qty_ep else np.nan,
        "tracc": exp_tot / qty_tot if qty_tot else np.nan,
        "share_exp_elec": exp_elec / exp_tot if exp_tot else np.nan,
        "share_exp_ep": exp_ep / exp_tot if exp_tot else np.nan,
    }

tracc: dict[str, pd.DataFrame] = {}
for case_key in CASE_FOLDERS:
    rows = []
    for risk in RISK_ORDER:
        if risk not in ao_full.get(case_key, {}) or risk not in prices.get(case_key, {}):
            continue
        try:
            r = compute_tracc(case_key, risk)
        except Exception as exc:
            print(f"TRACC skip {case_key}/{risk}: {type(exc).__name__}: {exc}")
            continue
        r.update(risk_folder=risk, risk_label=RISK_FOLDERS[risk])
        rows.append(r)
        print(f"TRACC {case_key:4s} | {risk:14s} | {r['tracc']:.2f} EUR/MWh")
    if rows:
        tracc[case_key] = pd.DataFrame(rows).set_index("risk_folder")
    else:
        print(f"TRACC skipped for {case_key} ({CASE_TITLES[case_key]}) — no loaded runs")

In [ ]:
# Cell 8 — shared plotting helpers (style, labels, tables)
def risk_xlabels():
    return RISK_LABELS

def fmt_num(v: float, digits: int = 1) -> str:
    if not np.isfinite(v):
        return "—"
    av = abs(v)
    if av >= 100:
        return f"{v:.0f}"
    if av >= 10:
        return f"{v:.1f}"
    if av >= 1:
        return f"{v:.{digits}f}"
    return f"{v:.2f}"

def style_ax(ax, ylabel: str | None = None, title: str | None = None):
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, loc="left", pad=8)
    ax.tick_params(axis="both", length=0)
    ax.grid(axis="y", color=C_GRID, linewidth=0.8)
    ax.grid(axis="x", visible=False)

def annotate_bars(ax, bars, values, *, fmt=None, dy_frac=0.02, fontsize=8.5, color=C_INK):
    """Put exact values on top of bars."""
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin if ymax > ymin else 1.0
    for bar, val in zip(bars, values):
        if not np.isfinite(val):
            continue
        txt = fmt(val) if fmt else fmt_num(val)
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + span * dy_frac,
            txt,
            ha="center", va="bottom",
            fontsize=fontsize, color=color, fontweight="medium",
            clip_on=False,
        )

def pad_ylim_for_labels(ax, values, low_pad=0.02, high_pad=0.18):
    vals = [v for v in values if np.isfinite(v)]
    if not vals:
        return
    lo, hi = min(vals), max(vals)
    if lo == hi:
        lo, hi = lo * 0.9, hi * 1.1 if hi else 1.0
    span = hi - lo
    ax.set_ylim(lo - low_pad * span, hi + high_pad * span)

def save_figure_and_table(fig, stem: str, table: pd.DataFrame) -> tuple[Path, Path]:
    png = FIG_DIR / f"{stem}.png"
    csv = TABLE_DIR / f"{stem}.csv"
    fig.savefig(png, bbox_inches="tight", facecolor="white")
    table.to_csv(csv, index=False, float_format="%.6g")
    print(f"saved {png.name}  +  tables/{csv.name}")
    return png, csv

def display_table(table: pd.DataFrame, title: str):
    print(f"\n— {title} —")
    with pd.option_context("display.max_columns", 20, "display.width", 120, "display.float_format", "{:.4g}".format):
        print(table.to_string(index=False))

def case_loaded(case_key: str) -> bool:
    return case_key in summary and not summary[case_key].empty

def skip_missing_case(case_key: str, what: str = "figure") -> bool:
    if case_loaded(case_key):
        return False
    print(f"Skipping {what} for {CASE_TITLES.get(case_key, case_key)} — no results loaded.")
    return True

def series_for(df: pd.DataFrame | None, col: str) -> list[float]:
    if df is None or df.empty:
        return [np.nan] * len(RISK_ORDER)
    return [float(df.loc[r, col]) if r in df.index else np.nan for r in RISK_ORDER]

def loaded_risks(case_key: str) -> list[str]:
    if case_key not in prices:
        return []
    return [r for r in RISK_ORDER if r in prices[case_key]]

print("Style helpers ready")

In [ ]:
# Cell 9 — figure functions
def plot_ammonia_cost_gap(case_key: str):
    if skip_missing_case(case_key, "ammonia cost gap"):
        return
    s = summary[case_key]
    labels = ["Grey\n(expected MC)"] + risk_xlabels()
    values = [GREY_MC_EXPECTED] + series_for(s, "cwap_EP")
    colors = [C_GREY] + [RISK_COLORS[r] for r in RISK_ORDER]
    rn_ep = float(s.loc["RN", "cwap_EP"]) if "RN" in s.index else np.nan

    table = pd.DataFrame({
        "series": ["Grey expected MC"] + [RISK_FOLDERS[r] for r in RISK_ORDER],
        "ammonia_EUR_per_MWh_EP": values,
        "vs_grey": [v - GREY_MC_EXPECTED for v in values],
        "vs_RN": [np.nan] + [v - rn_ep for v in series_for(s, "cwap_EP")],
    })

    fig, ax = plt.subplots(figsize=(9.5, 5.0))
    x = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, width=0.72, edgecolor="white", linewidth=0.8, zorder=3)
    ax.axhline(GREY_MC_EXPECTED, color=C_GREY, ls="--", lw=1.1, alpha=0.85, zorder=2)
    ax.set_xticks(x, labels)
    style_ax(ax, ylabel="€/MWh_EP", title=f"{CASE_TITLES[case_key]} — grey cost vs green ammonia CWAP")
    pad_ylim_for_labels(ax, values, low_pad=0.0, high_pad=0.22)
    ax.set_ylim(0, ax.get_ylim()[1])
    annotate_bars(ax, bars, values, fmt=lambda v: f"{v:.1f}")
    fig.tight_layout()
    save_figure_and_table(fig, f"{case_key}_01_ammonia_cost_gap", table)
    display_table(table, "Ammonia cost gap")
    plt.show()


def plot_system_cost_and_market_prices(case_key: str):
    if skip_missing_case(case_key, "system cost and market prices"):
        return
    s = summary[case_key]
    # Table: one row per risk, all series
    table = pd.DataFrame({
        "risk": risk_xlabels(),
        "system_cost_bn_EUR": series_for(s, "system_cost_bn"),
        **{f"wmean_{mkt}_EUR_per_MWh": series_for(s, f"wmean_{mkt}")
           for mkt in PRICE_COLS},
    })

    fig = plt.figure(figsize=(12.5, 7.2))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.05, 1.0], hspace=0.35, wspace=0.28)

    # System cost — full-width top-left span
    ax0 = fig.add_subplot(gs[0, 0])
    costs = table["system_cost_bn_EUR"].tolist()
    x = np.arange(len(RISK_ORDER))
    bars = ax0.bar(x, costs, color=[RISK_COLORS[r] for r in RISK_ORDER], width=0.7,
                   edgecolor="white", linewidth=0.8, zorder=3)
    ax0.set_xticks(x, risk_xlabels(), fontsize=8.5)
    style_ax(ax0, ylabel="bn €", title="Expected total system cost")
    pad_ylim_for_labels(ax0, costs, low_pad=0.15, high_pad=0.25)
    annotate_bars(ax0, bars, costs, fmt=lambda v: f"{v:.3f}", fontsize=7.5)

    # Five market W-means — each own axis so small GC prices stay readable
    market_axes = [
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[0, 2]),
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
        fig.add_subplot(gs[1, 2]),
    ]
    for ax, mkt in zip(market_axes, PRICE_COLS):
        vals = table[f"wmean_{mkt}_EUR_per_MWh"].tolist()
        bars = ax.bar(x, vals, color=MARKET_COLORS[mkt], width=0.7,
                      edgecolor="white", linewidth=0.8, zorder=3)
        ax.set_xticks(x, risk_xlabels(), fontsize=8)
        style_ax(ax, ylabel="€/MWh", title=MARKET_LABELS[mkt])
        pad_ylim_for_labels(ax, vals, low_pad=0.05, high_pad=0.28)
        annotate_bars(ax, bars, vals, fmt=lambda v: fmt_num(v, 2), fontsize=7)

    fig.suptitle(CASE_TITLES[case_key], fontsize=14, fontweight="semibold", y=0.98)
    save_figure_and_table(fig, f"{case_key}_02_system_cost_and_market_prices", table)
    display_table(table, "System cost & market W-mean prices")
    plt.show()


def plot_hourly_market_prices(case_key: str):
    risks = loaded_risks(case_key)
    if not risks:
        skip_missing_case(case_key, "hourly market prices")
        return
    rows = []
    fig, axes = plt.subplots(len(risks), 1, figsize=(12, max(2.7 * len(risks), 4.5)),
                             sharex=True, squeeze=False)
    axes = axes.ravel()
    t = np.arange(1, N_TS * N_RD * N_YR + 1)
    for ax, risk in zip(axes, risks):
        df = prices[case_key][risk]
        w = df["W"].to_numpy(float)
        n = min(len(t), len(df))
        for mkt, col in PRICE_COLS.items():
            y = df[col].to_numpy(float)
            ax.plot(t[:n], y[:n], lw=0.85, alpha=0.92, color=MARKET_COLORS[mkt], label=MARKET_LABELS[mkt])
            rows.append({
                "risk": RISK_FOLDERS[risk],
                "market": MARKET_LABELS[mkt],
                "wmean": float(np.average(y, weights=w)),
                "min": float(np.min(y)),
                "p10": float(np.quantile(y, 0.10)),
                "p50": float(np.quantile(y, 0.50)),
                "p90": float(np.quantile(y, 0.90)),
                "max": float(np.max(y)),
            })
        style_ax(ax, ylabel="€/MWh", title=RISK_FOLDERS[risk])
        ax.axhline(0, color="#CBD5E1", lw=0.7)
    axes[0].legend(ncol=5, loc="upper right", fontsize=8, bbox_to_anchor=(1.0, 1.18))
    axes[-1].set_xlabel("Time index (24 × 8 × 15 slots)")
    fig.suptitle(f"{CASE_TITLES[case_key]} — hourly market prices", fontsize=14,
                 fontweight="semibold", y=0.995)
    fig.tight_layout()
    table = pd.DataFrame(rows)
    save_figure_and_table(fig, f"{case_key}_03_hourly_market_prices", table)
    display_table(table, "Hourly price summary (percentiles)")
    plt.show()


def price_duration(df: pd.DataFrame, col: str):
    p = df[col].to_numpy(float)
    w = df["W"].to_numpy(float)
    order = np.argsort(-p)
    p_sorted = p[order]
    w_sorted = w[order]
    cum = np.cumsum(w_sorted)
    return cum / cum[-1], p_sorted

def plot_price_duration_curves(case_key: str):
    risks = loaded_risks(case_key)
    if not risks:
        skip_missing_case(case_key, "price duration curves")
        return
    markets = ["elec", "H2", "EP"]
    fracs = [0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0]
    rows = []
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.6))
    for ax, mkt in zip(axes, markets):
        for risk in risks:
            x, y = price_duration(prices[case_key][risk], PRICE_COLS[mkt])
            ax.plot(x, y, color=RISK_COLORS[risk], lw=2.0, label=RISK_FOLDERS[risk])
            for f in fracs:
                rows.append({
                    "market": MARKET_LABELS[mkt],
                    "risk": RISK_FOLDERS[risk],
                    "duration_fraction": f,
                    "price_EUR_per_MWh": float(np.interp(f, x, y)),
                })
        style_ax(ax, ylabel="€/MWh" if mkt == "elec" else None, title=MARKET_LABELS[mkt])
        ax.set_xlabel("Share of weighted hours")
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
        ax.set_xlim(0, 1)
        ax.axhline(0, color="#CBD5E1", lw=0.7)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{CASE_TITLES[case_key]} — price duration curves", fontsize=14,
                 fontweight="semibold", y=1.02)
    fig.tight_layout()
    table = pd.DataFrame(rows)
    save_figure_and_table(fig, f"{case_key}_04_price_duration_curves", table)
    display_table(table.head(20), "Price duration (first 20 rows; full CSV saved)")
    plt.show()


def plot_investments(case_key: str):
    """One panel per technology with its own scale (GW vs MW)."""
    if skip_missing_case(case_key, "investments"):
        return
    tab = inv_tables[case_key]
    table = pd.DataFrame({
        "risk": risk_xlabels(),
        **{
            f"{INVESTABLE_LABELS[aid]}_{INVESTABLE_UNITS[aid]}":
                [v * INVESTABLE_SCALE[aid] for v in series_for(tab, aid)]
            for aid in INVESTABLE
        },
        **{f"{aid}_MW_raw": series_for(tab, aid) for aid in INVESTABLE},
    })

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.0), sharex=True)
    x = np.arange(len(RISK_ORDER))
    for ax, aid in zip(axes.ravel(), INVESTABLE):
        scale = INVESTABLE_SCALE[aid]
        unit = INVESTABLE_UNITS[aid]
        vals = series_for(tab, aid)
        vals = [v * scale for v in vals]
        bars = ax.bar(
            x, vals, width=0.72, color=INV_COLORS[aid],
            edgecolor="white", linewidth=0.8, zorder=3,
        )
        ax.set_xticks(x, risk_xlabels())
        style_ax(ax, ylabel=unit, title=INVESTABLE_LABELS[aid])
        pad_ylim_for_labels(ax, vals, low_pad=0.0, high_pad=0.28)
        ax.set_ylim(0, ax.get_ylim()[1])
        annotate_bars(
            ax, bars, vals,
            fmt=(lambda v: f"{v:.2f}") if unit == "GW" else (lambda v: f"{v:.0f}"),
            fontsize=8,
        )

    fig.suptitle(f"{CASE_TITLES[case_key]} — new investment by technology",
                 fontsize=14, fontweight="semibold", y=0.98)
    fig.tight_layout()
    save_figure_and_table(fig, f"{case_key}_05_investments", table)
    display_table(table, "Investments")
    plt.show()


def plot_tracc(case_key: str):
    if case_key not in tracc or tracc[case_key].empty:
        skip_missing_case(case_key, "TRACC")
        return
    t = tracc[case_key]
    table = pd.DataFrame({
        "risk": risk_xlabels(),
        "TRACC_EUR_per_MWh": series_for(t, "tracc"),
        "elec_CWAP_EUR_per_MWh": series_for(t, "cwap_elec"),
        "EP_CWAP_EUR_per_MWh": series_for(t, "cwap_ep"),
        "share_exp_elec": series_for(t, "share_exp_elec"),
        "share_exp_ep": series_for(t, "share_exp_ep"),
        "exp_elec_EUR": series_for(t, "exp_elec"),
        "exp_ep_EUR": series_for(t, "exp_ep"),
    })

    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.8))
    x = np.arange(len(RISK_ORDER))

    ax = axes[0]
    y_tracc = table["TRACC_EUR_per_MWh"].tolist()
    y_el = table["elec_CWAP_EUR_per_MWh"].tolist()
    y_ep = table["EP_CWAP_EUR_per_MWh"].tolist()
    ax.plot(x, y_tracc, "-o", color=C_INK, lw=2.2, markersize=7, label="TRACC (elec+EP)", zorder=4)
    ax.plot(x, y_el, "--s", color=MARKET_COLORS["elec"], lw=1.5, markersize=6, label="Elec CWAP")
    ax.plot(x, y_ep, "--^", color=MARKET_COLORS["EP"], lw=1.5, markersize=6, label="Ammonia CWAP")
    for i, v in enumerate(y_tracc):
        if not np.isfinite(v):
            continue
        ax.annotate(f"{v:.1f}", (x[i], v), textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8, color=C_INK, fontweight="medium")
    ax.set_xticks(x, risk_xlabels())
    style_ax(ax, ylabel="€ / MWh final energy", title="Total risk-adjusted consumer cost")
    pad_ylim_for_labels(ax, y_tracc + y_el + y_ep, low_pad=0.08, high_pad=0.22)
    ax.legend(loc="best")

    ax = axes[1]
    elec_share = table["share_exp_elec"].tolist()
    ep_share = table["share_exp_ep"].tolist()
    b1 = ax.bar(x, elec_share, color=MARKET_COLORS["elec"], width=0.66,
                label="Electricity", edgecolor="white", linewidth=0.8, zorder=3)
    b2 = ax.bar(x, ep_share, bottom=elec_share, color=MARKET_COLORS["EP"], width=0.66,
                label="Ammonia", edgecolor="white", linewidth=0.8, zorder=3)
    for i, (e, p) in enumerate(zip(elec_share, ep_share)):
        if not (np.isfinite(e) and np.isfinite(p)):
            continue
        ax.text(i, e / 2, f"{100*e:.0f}%", ha="center", va="center",
                fontsize=8, color="white", fontweight="semibold")
        ax.text(i, e + p / 2, f"{100*p:.0f}%", ha="center", va="center",
                fontsize=8, color="white", fontweight="semibold")
    ax.set_xticks(x, risk_xlabels())
    ax.set_ylim(0, 1.08)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    style_ax(ax, ylabel="Share of expenditure", title="Final-energy expenditure mix")
    ax.legend(loc="upper right")

    fig.suptitle(
        f"{CASE_TITLES[case_key]} — TRACC (no intermediate H₂)",
        fontsize=14, fontweight="semibold", y=1.02,
    )
    fig.tight_layout()
    save_figure_and_table(fig, f"{case_key}_06_tracc", table)
    display_table(table, "TRACC")
    plt.show()


def run_all_figures(case_key: str):
    if skip_missing_case(case_key, "all figures"):
        return
    print("=" * 64)
    print(CASE_TITLES[case_key])
    print("=" * 64)
    plot_ammonia_cost_gap(case_key)
    plot_system_cost_and_market_prices(case_key)
    plot_hourly_market_prices(case_key)
    plot_price_duration_curves(case_key)
    plot_investments(case_key)
    plot_tracc(case_key)

print("Figure functions ready")

---
## Social planner (SP)


In [ ]:
# Cell 10 — SP: ammonia cost gap
plot_ammonia_cost_gap("sp")

In [ ]:
# Cell 11 — SP: system cost + market prices
plot_system_cost_and_market_prices("sp")

In [ ]:
# Cell 12 — SP: hourly market prices
plot_hourly_market_prices("sp")

In [ ]:
# Cell 13 — SP: price duration curves
plot_price_duration_curves("sp")

In [ ]:
# Cell 14 — SP: investments
plot_investments("sp")

In [ ]:
# Cell 15 — SP: TRACC
plot_tracc("sp")

---
## Market exposure (ME)


In [ ]:
# Cell 16 — ME: ammonia cost gap
plot_ammonia_cost_gap("me")

In [ ]:
# Cell 17 — ME: system cost + market prices
plot_system_cost_and_market_prices("me")

In [ ]:
# Cell 18 — ME: hourly market prices
plot_hourly_market_prices("me")

In [ ]:
# Cell 19 — ME: price duration curves
plot_price_duration_curves("me")

In [ ]:
# Cell 20 — ME: investments
plot_investments("me")

In [ ]:
# Cell 21 — ME: TRACC
plot_tracc("me")

---
## Green social planner (G-SP)


In [ ]:
# Cell 22 — GSP: ammonia cost gap
plot_ammonia_cost_gap("gsp")

In [ ]:
# Cell 23 — GSP: system cost + market prices
plot_system_cost_and_market_prices("gsp")

In [ ]:
# Cell 24 — GSP: hourly market prices
plot_hourly_market_prices("gsp")

In [ ]:
# Cell 25 — GSP: price duration curves
plot_price_duration_curves("gsp")

In [ ]:
# Cell 26 — GSP: investments
plot_investments("gsp")

In [ ]:
# Cell 27 — GSP: TRACC
plot_tracc("gsp")

---
## Green H₂ social planner (GH2-SP)


In [ ]:
# Cell 28 — GH2: ammonia cost gap
plot_ammonia_cost_gap("gh2")

In [ ]:
# Cell 29 — GH2: system cost + market prices
plot_system_cost_and_market_prices("gh2")

In [ ]:
# Cell 30 — GH2: hourly market prices
plot_hourly_market_prices("gh2")

In [ ]:
# Cell 31 — GH2: price duration curves
plot_price_duration_curves("gh2")

In [ ]:
# Cell 32 — GH2: investments
plot_investments("gh2")

In [ ]:
# Cell 33 — GH2: TRACC
plot_tracc("gh2")

---
## Cross-case comparison


In [ ]:
# Cell 34 — TRACC across available market designs
cases = [ck for ck in CASE_FOLDERS if ck in tracc and not tracc[ck].empty]
if not cases:
    print("Skipping cross-case TRACC — no cases loaded.")
else:
    table = pd.DataFrame({
        "risk": risk_xlabels(),
        **{CASE_TITLES[ck]: series_for(tracc[ck], "tracc") for ck in cases},
    })

    fig, ax = plt.subplots(figsize=(9.5, 5.0))
    x = np.arange(len(RISK_ORDER))
    all_vals = []
    for ck in cases:
        vals = table[CASE_TITLES[ck]].tolist()
        all_vals.extend(vals)
        ax.plot(x, vals, "-o", color=CASE_COLORS[ck], lw=2.1, markersize=6.5, label=CASE_TITLES[ck])
        for i, v in enumerate(vals):
            if not np.isfinite(v):
                continue
            ax.annotate(f"{v:.1f}", (x[i], v), textcoords="offset points",
                        xytext=(0, 7), ha="center", fontsize=7.5, color=CASE_COLORS[ck])
    ax.set_xticks(x, risk_xlabels())
    style_ax(ax, ylabel="TRACC (€ / MWh final energy)", title="Total risk-adjusted consumer cost — loaded cases")
    pad_ylim_for_labels(ax, all_vals, high_pad=0.2)
    ax.legend()
    fig.tight_layout()
    save_figure_and_table(fig, "all_cases_tracc", table)
    display_table(table, "Cross-case TRACC")
    plt.show()
    missing = [CASE_TITLES[ck] for ck in CASE_FOLDERS if ck not in cases]
    if missing:
        print("Not yet generated:", ", ".join(missing))

In [ ]:
# Cell 35 — wind investment across loaded cases
cases = [ck for ck in CASE_FOLDERS if ck in inv_tables and not inv_tables[ck].empty]
if not cases:
    print("Skipping cross-case wind investment — no cases loaded.")
else:
    table = pd.DataFrame({
        "risk": risk_xlabels(),
        **{CASE_TITLES[ck]: [v / 1e3 for v in series_for(inv_tables[ck], "Gen_VRES_Wind")]
           for ck in cases},
    })

    fig, ax = plt.subplots(figsize=(10, 5.0))
    x = np.arange(len(RISK_ORDER))
    n = len(cases)
    width = 0.18 if n >= 4 else min(0.22, 0.7 / max(n, 1))
    offset0 = (n - 1) / 2
    all_vals = []
    for i, ck in enumerate(cases):
        vals = table[CASE_TITLES[ck]].tolist()
        all_vals.extend(vals)
        bars = ax.bar(x + (i - offset0) * width, vals, width=width, color=CASE_COLORS[ck],
                      label=CASE_TITLES[ck], edgecolor="white", linewidth=0.6, zorder=3)
        annotate_bars(ax, bars, vals, fmt=lambda v: f"{v:.1f}", fontsize=6.5, dy_frac=0.015)
    ax.set_xticks(x, risk_xlabels())
    style_ax(ax, ylabel="GW", title="Wind investment across market designs")
    pad_ylim_for_labels(ax, all_vals, low_pad=0.0, high_pad=0.22)
    ax.set_ylim(0, ax.get_ylim()[1])
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure_and_table(fig, "all_cases_wind_investment", table)
    display_table(table, "Wind investment (GW)")
    plt.show()
    missing = [CASE_TITLES[ck] for ck in CASE_FOLDERS if ck not in cases]
    if missing:
        print("Not yet generated:", ", ".join(missing))

---
## Optional — regenerate everything


In [ ]:
# Cell 36 — run all figure blocks for every loaded case
for ck in CASE_FOLDERS:
    run_all_figures(ck)
print("Done.")
print("Figures:", FIG_DIR)
print("Tables: ", TABLE_DIR)
if skipped_runs:
    print(f"Skipped runs ({len(skipped_runs)}):")
    for line in skipped_runs:
        print(f"  - {line}")